In [ ]:
REPO_URL = "https://github.com/boldirev-as/hw05.git"
CHECKPOINT_URL = "https://github.com/boldirev-as/hw05/releases/download/checkpoint-v1/best.pt"
WORK_DIR = "/content"
DATA_ZIP_URL = "https://github.com/boldirev-as/hw05/releases/download/checkpoint-v1/demo_data.zip"
DATA_DIR = f"{WORK_DIR}/demo_data"
RECON_DIR = f"{WORK_DIR}/recon"

In [ ]:
!mkdir -p "$WORK_DIR"
!rm -rf "$WORK_DIR/hw05"
!git clone $REPO_URL "$WORK_DIR/hw05"
%cd $WORK_DIR/hw05
!pip install -q -r requirements.txt

In [ ]:
!wget -q -O "$WORK_DIR/best.pt" "$CHECKPOINT_URL"
!rm -rf "$DATA_DIR" "$WORK_DIR/data.zip"
!wget -q -O "$WORK_DIR/data.zip" "$DATA_ZIP_URL"
!unzip -q -o "$WORK_DIR/data.zip" -d "$WORK_DIR"

In [ ]:
!find "$DATA_DIR" -maxdepth 2 -type f | sort


In [ ]:
!python inference.py --steps 5 --ckpt "$WORK_DIR/best.pt" --data "$DATA_DIR" --out "$RECON_DIR" --size 256 --base 64
!find "$RECON_DIR" -maxdepth 1 -type f | sort

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

data = Path(DATA_DIR)
pred = Path(RECON_DIR)
ids = [p.stem for p in sorted(pred.glob('*.png'))[:4]]
for name in ids:
    images = [
        Image.open(data / 'lensed' / f'{name}.png').convert('RGB'),
        Image.open(data / 'lensless' / f'{name}.png').convert('RGB'),
        Image.open(pred / f'{name}.png').convert('RGB'),
    ]
    titles = ['оригинал', 'безлинзовое', 'реконструкция']
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis('off')
    plt.show()

In [ ]:
!python calculate_metrics.py --gt "$DATA_DIR/lensed" --pred "$RECON_DIR" --size 256